# Fine-tuning модели с Hugging Face для узкой задачи

### Информация о датасете

Fine-tuning производится на русскоязычной части датасета [MLSUM - Multilingual Summarization](https://www.kaggle.com/datasets/thedevastator/mlsam-multilingual-summarization-dataset), предназначенном для задачи суммаризации текста на 5 языках: русском, французском, немецком, испанском и турецком и содержащем следующие данные для каждого языка:
- `text` - основная часть статьи (текст),
- `summary` - краткое изложение статьи (текст),
- `topic` - тема или категория статьи (текст),
- `url` - URL-адрес статьи (текст),
- `title` - название статьи (текст),
- `date` - дата публикации статьи (дата).

Подвыборки `train`, `val` и `test` русскоязычной части датасета содержит 25556, 750 и 757 записей соответственно.


### Предобработка датасета

Загрузим русские тексты - файлы `ru_train.csv`, `ru_validation.csv`, `ru_test.csv`. Поскольку для задачи сжатия текста актуальны только исходный текст статьи и ее саммари и могут быть полезны название и тема, очистим данные от лишних столбцов (URL и даты).

In [3]:
import numpy as np
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from transformers.trainer import Trainer
from transformers.training_args import TrainingArguments
import matplotlib.pyplot as plt

device = "cuda"  if torch.cuda.is_available() else "cpu"

In [4]:
rusum_dataset = load_dataset("reciTAL/mlsum", "ru")
rusum_dataset.remove_columns(["url", "date"])

README.md:   0%|          | 0.00/11.0k [00:00<?, ?B/s]

mlsum.py:   0%|          | 0.00/3.72k [00:00<?, ?B/s]

The repository for reciTAL/mlsum contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/reciTAL/mlsum.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N]  y


Generating train split:   0%|          | 0/25556 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/750 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/757 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'summary', 'topic', 'title'],
        num_rows: 25556
    })
    validation: Dataset({
        features: ['text', 'summary', 'topic', 'title'],
        num_rows: 750
    })
    test: Dataset({
        features: ['text', 'summary', 'topic', 'title'],
        num_rows: 757
    })
})

In [5]:
model_name = "cointegrated/rut5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)

tokenizer_config.json:   0%|          | 0.00/1.80k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/666 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/640k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/98.0 [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
/usr/local/lib/python3.11/dist-packages/transformers/convert_slow_tokenizer.py:559: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


model.safetensors:   0%|          | 0.00/259M [00:00<?, ?B/s]

In [6]:
def dataset_examples_to_tokens(examples):
    inputs = [f"Title: {title}. \n\nTopic: {topic}. \n\nText: {text}\n\n\n"
              for title, topic, text in zip(examples["title"], examples["topic"], examples["text"])]
    targets = examples["summary"]
    model_inputs = tokenizer(inputs, max_length=1024, padding="max_length", truncation=True)
    labels = tokenizer(targets, max_length=256, padding="max_length", truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_datasets = rusum_dataset.map(dataset_examples_to_tokens, batched=True)

Map:   0%|          | 0/25556 [00:00<?, ? examples/s]

Map:   0%|          | 0/750 [00:00<?, ? examples/s]

Map:   0%|          | 0/757 [00:00<?, ? examples/s]

In [22]:
training_args = TrainingArguments(
    output_dir="./rut5_mlsum_finetuned",
    eval_strategy="epoch",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    logging_steps=10,
    report_to="none",
    disable_tqdm=False,
    fp16=True,
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
)

In [23]:
train_metrics = trainer.train()
val_metrics = trainer.evaluate()
print("Train metrics:    ", train_metrics)
print("Final val metrics:", val_metrics)

/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Epoch,Training Loss,Validation Loss
1,0.364900,0.293984
2,0.364500,0.288892
3,0.376300,0.288237


/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked t

Train metrics:     TrainOutput(global_step=4794, training_loss=0.3881282561711585, metrics={'train_runtime': 5372.7396, 'train_samples_per_second': 14.27, 'train_steps_per_second': 0.892, 'total_flos': 2.560283339076403e+16, 'train_loss': 0.3881282561711585, 'epoch': 3.0})
Final val metrics: {'eval_loss': 0.2882366180419922, 'eval_runtime': 20.1124, 'eval_samples_per_second': 37.29, 'eval_steps_per_second': 1.193, 'epoch': 3.0}
